In [ ]:
# 1. Clone repository & system dependencies
%cd /content
!rm -rf /content/subtitlesssss
!git clone https://github.com/aananyaph/subtitlesssss
%cd /content/subtitlesssss

!apt-get update -qq
!apt-get install -y -qq libgl1 libegl1 libxkbcommon0 libdbus-1-3 libfontconfig1
!pip install -q -r backend/requirements.txt PyQt6

# Compatibility edits from the original repository
!sed -i 's/^from typing import Optional$/from typing import Any, Callable, Dict, Optional/' main_app.py
!sed -i '0,/^import logging$/s//from __future__ import annotations\nimport logging/' app/loopback_capture.py


In [ ]:
# 2. Install required IndicTrans2, ONNX-GPU, and benchmarking packages
!pip install -q indictranstoolkit sentencepiece sacremoses sacrebleu tokenizers indic-nlp-library pyngrok
!pip uninstall -y -q onnxruntime onnxruntime-gpu
!pip install -q onnxruntime-gpu


In [ ]:
# 3. Verify CUDA, PyTorch, and ONNX Runtime GPU Providers
!nvidia-smi

import os
import torch
import onnxruntime as ort
from google.colab import userdata

# Hugging Face token from Colab secrets
try:
    _hf_token = userdata.get('HF_TOKEN')
    if _hf_token:
        os.environ["HF_TOKEN"] = _hf_token.strip()
        print("✅ Successfully loaded HF_TOKEN from Colab Secrets.")
    else:
        print("⚠️ HF_TOKEN not found in Colab Secrets. Please add it under the '🔑' tab with the name 'HF_TOKEN'.")
except Exception as e:
    print(f"Could not load HF_TOKEN from secrets: {e}")

# Assert CUDA availability for PyTorch
print("PyTorch version:", torch.__version__)
assert torch.cuda.is_available(), (
    "CUDA is unavailable in PyTorch! Please ensure this notebook is running on a GPU runtime (Runtime -> Change runtime type -> T4 GPU)."
)
print("✅ PyTorch CUDA GPU:", torch.cuda.get_device_name(0))

# Assert CUDA availability for ONNX Runtime
print("ONNX Runtime version:", ort.__version__)
print("ONNX Runtime available providers:", ort.get_available_providers())
assert "CUDAExecutionProvider" in ort.get_available_providers(), (
    "CUDAExecutionProvider is unavailable in ONNX Runtime! Verify GPU runtime and onnxruntime-gpu compatibility."
)
print("✅ ONNX Runtime CUDA provider verified.")

from IndicTransToolkit import IndicProcessor
ip = IndicProcessor(inference=True)
print("✅ IndicProcessor successfully initialized.")


In [ ]:
# 4. Optional: Mount Google Drive to persist HF model cache across Colab runtime resets
import os

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    drive_cache = "/content/drive/MyDrive/huggingface_cache"
    os.makedirs(drive_cache, exist_ok=True)
    os.environ["HF_HOME"] = drive_cache
    print("✅ HF_HOME set to Google Drive:", drive_cache)
except Exception as e:
    print("ℹ️ Google Drive mount skipped/failed, using local runtime cache:", e)


In [ ]:
# 5. Pre-download IndicTrans2 1B repos and IndicConformer
import os, shutil
from huggingface_hub import snapshot_download

hf_token = os.environ.get("HF_TOKEN") or None

# Pre-download IndicConformer for Indic ASR
print("Downloading IndicConformer ASR model...")
snapshot_download("ai4bharat/indic-conformer-600m-multilingual", token=hf_token)

root = "/root/.cache/huggingface/hub/models--ai4bharat--indic-conformer-600m-multilingual/snapshots"
if os.path.exists(root):
    copied = 0
    for dp, _, filenames in os.walk(root):
        for filename in filenames:
            file_path = os.path.join(dp, filename)
            if os.path.islink(file_path):
                target = os.path.realpath(file_path)
                if not os.path.exists(target):
                    raise FileNotFoundError(f"Broken symlink: {file_path} -> {target}")
                os.remove(file_path)
                shutil.copy2(target, file_path)
                copied += 1
    print("Replaced symlinks with file copies for IndicConformer:", copied)

# Pre-download IndicTrans2 1B translation models
repos_1b = [
    "ai4bharat/indictrans2-en-indic-1B",
    "ai4bharat/indictrans2-indic-en-1B",
    "ai4bharat/indictrans2-indic-indic-1B",
]

for repo in repos_1b:
    try:
        print(f"Pre-downloading/verifying: {repo}...")
        snapshot_download(repo, token=hf_token)
        print(f"✅ Ready: {repo}")
    except Exception as e:
        print(f"⚠️ Could not download {repo}: {e}")


In [ ]:
# 6. Apply dynamic ASR config patch and extract ONNX fallback translator
path = "/content/subtitlesssss/backend/app_bridge.py"
with open(path, "r", encoding="utf-8") as f:
    s = f.read()

if "def apply_asr_config" not in s:
    helper = '''# ---------- Dynamic ASR config (from client HANDSHAKE) ----------
_current_cfg_key = None

def apply_asr_config(cfg):
    """Rebuild pipeline to match client's audio/subtitle language choice."""
    global pipeline, _current_cfg_key
    key = (
        cfg.get("model_name") or "base",
        cfg.get("indic_lang") or None,
        cfg.get("asr_language") or None,
        cfg.get("asr_task") or None,
    )
    if pipeline is not None and key == _current_cfg_key:
        logging.info(f"[Bridge] ASR config unchanged {key}; keeping pipeline")
        return
    logging.info(f"[Bridge] Switching ASR config -> model={key[0]} indic={key[1]} lang={key[2]} task={key[3]}")
    import main_app
    old = pipeline
    if old is not None:
        try:
            old.stop()
        except Exception as e:
            logging.warning(f"[Bridge] old pipeline stop failed: {e}")
    if key[1]:
        from indic_backend import load_indic_model
        load_indic_model()
    new = main_app.DesktopLiveSubtitlePipeline(
        overlay=BridgeOverlay(),
        model_name=key[0],
        indic_lang=key[1],
        asr_language=key[2],
        asr_task=key[3],
    )
    try:
        new.start()
    except Exception as e:
        logging.warning(f"[Bridge] new pipeline start failed (no audio device?) {e}")
    pipeline = new
    _current_cfg_key = key
    logging.info("[Bridge] ASR config applied")

'''
    marker = "# ---------- WebSocket handler ----------"
    s = s.replace(marker, helper + marker)
    old = '                    logging.info(f"[Bridge] HANDSHAKE {data}")\n'
    new = '''                    logging.info(f"[Bridge] HANDSHAKE {data}")
                    _cfg = data.get("asrConfig")
                    if _cfg:
                        try:
                            await asyncio.get_running_loop().run_in_executor(None, apply_asr_config, _cfg)
                        except Exception as e:
                            logging.error(f"[Bridge] apply_asr_config failed: {e}", exc_info=True)
                            await ws.send(json.dumps({"type": "final", "text": f"Server model load failed: {e}", "is_final": True}))
'''
    s = s.replace(old, new)
    with open(path, "w", encoding="utf-8") as f:
        f.write(s)
    print("app_bridge.py patched: dynamic ASR config on HANDSHAKE")

# Patch main_app.py for Kannada unicode
path_main = "/content/subtitlesssss/main_app.py"
with open(path_main, "r", encoding="utf-8") as f:
    s_main = f.read()

if "_kn_fix" not in s_main:
    old_m = '                text = " ".join(s["text"] for s in result.get("segments", [])).strip()\n'
    new_m = '''                text = " ".join(s["text"] for s in result.get("segments", [])).strip()
                if self.indic_lang == "kn" and text:
                    # _kn_fix: stray Tamil block (U+0B80-0BFF) -> Kannada block (U+0C80-0CFF)
                    text = "".join(
                        chr(ord(c) - 0x0B80 + 0x0C80) if 0x0B80 <= ord(c) <= 0x0BFF else c
                        for c in text
                    )
'''
    s_main = s_main.replace(old_m, new_m)
    with open(path_main, "w", encoding="utf-8") as f:
        f.write(s_main)
    print("main_app.py patched: kn script sanitize in Indic path")

# Extract translator_live.py (ONNX fallback) if needed
import ast
root = "/content/subtitlesssss"
if not os.path.exists(f"{root}/translator_live.py"):
    src = open(f"{root}/pipeline.py", encoding="utf-8").read()
    lines = src.splitlines(keepends=True)
    chunks = []
    for node in ast.parse(src).body:
        if isinstance(node, ast.Assign) and any(getattr(t, "id", None) == "LANG_MAP" for t in node.targets):
            chunks.append("".join(lines[node.lineno - 1:node.end_lineno]))
        elif isinstance(node, ast.ClassDef) and node.name == "IndicTransONNXModel":
            chunks.append("".join(lines[node.lineno - 1:node.end_lineno]))
    header = '''import json, logging
from pathlib import Path
import numpy as np
import onnxruntime as ort
from huggingface_hub import snapshot_download
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_detokenize
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator
from sacremoses import MosesDetokenizer, MosesTokenizer
from tokenizers import Tokenizer

logger = logging.getLogger("translator_live")

'''
    open(f"{root}/translator_live.py", "w", encoding="utf-8").write(header + "".join(chunks))
    p = f"{root}/translator_live.py"
    t = open(p, encoding="utf-8").read()
    old_l = "        self.snap = Path(snapshot_download(repo_id))\n"
    new_l = old_l + '''        import os, shutil  # _symlink_fix
        for _dp, _, _fns in os.walk(self.snap):
            for _fn in _fns:
                _fp = os.path.join(_dp, _fn)
                if os.path.islink(_fp):
                    _t = os.path.realpath(_fp)
                    os.remove(_fp)
                    shutil.copy2(_t, _fp)
'''
    if "_symlink_fix" not in t and old_l in t:
        open(p, "w", encoding="utf-8").write(t.replace(old_l, new_l))
    print("translator_live.py ready")


In [ ]:
# 7. STEP 4: Direct Model Benchmark (6 Directions, 20 Sentences Each)
import gc
import json
import os
import sys
import time
import numpy as np

sys.path.insert(0, "/content/subtitlesssss")

import sacrebleu
import torch
import translation_proxy as tp

# 20 parallel test sentences across English, Hindi, and Kannada
test_sentences = {
    "en": [
        "Hello, how are you today?",
        "Good morning to everyone present here.",
        "The weather is very pleasant this evening.",
        "Where is the nearest railway station?",
        "Can you please help me with this problem?",
        "We are developing a real-time subtitle translation system.",
        "Artificial intelligence is transforming communication across the world.",
        "I enjoy listening to classical Indian music while working.",
        "Please make sure to submit your assignment before Friday.",
        "The train will arrive at platform number three in ten minutes.",
        "India is a country with rich cultural heritage and diversity.",
        "What time does the library close on weekends?",
        "Health and education are fundamental rights for every citizen.",
        "She works as a software engineer in Bengaluru.",
        "Can you recommend a good book to read?",
        "Water conservation is essential for our future generations.",
        "The children are playing happily in the garden.",
        "Scientists have discovered a new species in the Western Ghats.",
        "Learning multiple languages expands cognitive abilities and empathy.",
        "Thank you very much for your time and guidance."
    ],
    "hi": [
        "नमस्ते, आप आज कैसे हैं?",
        "यहाँ उपस्थित सभी लोगों को सुप्रभात।",
        "आज शाम का मौसम बहुत सुहावना है।",
        "निकटतम रेलवे स्टेशन कहाँ है?",
        "क्या आप इस समस्या में मेरी मदद कर सकते हैं?",
        "हम एक वास्तविक समय उपशीर्षक अनुवाद प्रणाली विकसित कर रहे हैं।",
        "कृत्रिम बुद्धिमत्ता दुनिया भर में संचार को बदल रही है।",
        "मुझे काम करते समय भारतीय शास्त्रीय संगीत सुनना पसंद है।",
        "कृपया शुक्रवार से पहले अपना असाइनमेंट जमा करना सुनिश्चित करें।",
        "ट्रेन दस मिनट में प्लेटफॉर्म नंबर तीन पर पहुंचेगी।",
        "भारत समृद्ध सांस्कृतिक विरासत और विविधता वाला देश है।",
        "सप्ताहांत पर पुस्तकालय किस समय बंद होता है?",
        "स्वास्थ्य और शिक्षा प्रत्येक नागरिक का मौलिक अधिकार है।",
        "वह बेंगलुरु में एक सॉफ्टवेयर इंजीनियर के रूप में काम करती है।",
        "क्या आप पढ़ने के लिए किसी अच्छी किताब की सिफारिश कर सकते हैं?",
        "जल संरक्षण हमारी भावी पीढ़ियों के लिए आवश्यक है।",
        "बच्चे बगीचे में खुशी से खेल रहे हैं।",
        "वैज्ञानिकों ने पश्चिमी घाट में एक नई प्रजाति की खोज की है।",
        "कई भाषाएं सीखने से सोचने-समझने की क्षमता और संवेदनशीलता बढ़ती है।",
        "आपके समय और मार्गदर्शन के लिए बहुत-बहुत धन्यवाद।"
    ],
    "kn": [
        "ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?",
        "ಇಲ್ಲಿ ನೆರೆದಿರುವ ಎಲ್ಲರಿಗೂ ಶುಭೋದಯ.",
        "ಇಂದು ಸಂಜೆಯ ವಾತಾವರಣ ತುಂಬಾ ಆಹ್ಲಾದಕರವಾಗಿದೆ.",
        "ಹತ್ತಿರದ ರೈಲ್ವೆ ನಿಲ್ದಾಣ ಎಲ್ಲಿದೆ?",
        "ದಯವಿಟ್ಟು ಈ ಸಮಸ್ಯೆಯಲ್ಲಿ ನನಗೆ ಸಹಾಯ ಮಾಡಬಹುದೇ?",
        "ನಾವು ನೈಜ ಸಮಯದ ಉಪಶೀರ್ಷಿಕೆ ಅನುವಾದ ವ್ಯವಸ್ಥೆಯನ್ನು ಅಭಿವೃದ್ಧಿಪಡಿಸುತ್ತಿದ್ದೇವೆ.",
        "ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆಯು ಪ್ರಪಂಚದಾದ್ಯಂತ ಸಂವಹನವನ್ನು ಬದಲಾಯಿಸುತ್ತಿದೆ.",
        "ಕೆಲಸ ಮಾಡುವಾಗ ಭಾರತೀಯ ಶಾಸ್ತ್ರೀಯ ಸಂಗೀತವನ್ನು ಕೇಳಲು ನಾನು ಇಷ್ಟಪಡುತ್ತೇನೆ.",
        "ದಯವಿಟ್ಟು ಶುಕ್ರವಾರದ ಮೊದಲು ನಿಮ್ಮ ನಿಯೋಜನೆಯನ್ನು ಸಲ್ಲಿಸಲು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ.",
        "ರೈಲು ಹತ್ತು ನಿಮಿಷಗಳಲ್ಲಿ ಪ್ಲಾಟ್‌ಫಾರ್ಮ್ ಸಂಖ್ಯೆ ಮೂರಕ್ಕೆ ಬರಲಿದೆ.",
        "ಭಾರತವು ಶ್ರೀಮಂತ ಸಾಂಸ್ಕೃತಿಕ ಪರಂಪರೆ ಮತ್ತು ವೈವಿಧ್ಯತೆಯನ್ನು ಹೊಂದಿರುವ ದೇಶವಾಗಿದೆ.",
        "ವಾರಾಂತ್ಯದಲ್ಲಿ ಗ್ರಂಥಾಲಯವು ಯಾವ ಸಮಯಕ್ಕೆ ಮುಚ್ಚುತ್ತದೆ?",
        "ಆರೋಗ್ಯ ಮತ್ತು ಶಿಕ್ಷಣ ಪ್ರತಿಯೊಬ್ಬ ನಾಗರಿಕನ ಮೂಲಭೂತ ಹಕ್ಕು.",
        "ಅವರು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಸಾಫ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರ್ ಆಗಿ ಕೆಲಸ ಮಾಡುತ್ತಿದ್ದಾರೆ.",
        "ಓದಲು ಯಾವುದಾದರೂ ಒಳ್ಳೆಯ ಪುಸ್ತಕವನ್ನು ನೀವು ಶಿಫಾರಸು ಮಾಡಬಹುದೇ?",
        "ನಮ್ಮ ಮುಂದಿನ ಪೀಳಿಗೆಗೆ ಜಲ ಸಂರಕ್ಷಣೆ ಅತ್ಯಗತ್ಯ.",
        "ಮಕ್ಕಳು ತೋಟದಲ್ಲಿ ಸಂತೋಷದಿಂದ ಆಟವಾಡುತ್ತಿದ್ದಾರೆ.",
        "ವಿಜ್ಞಾನಿಗಳು ಪಶ್ಚಿಮ ಘಟ್ಟಗಳಲ್ಲಿ ಹೊಸ ಜಾತಿಯನ್ನು ಕಂಡುಹಿಡಿದಿದ್ದಾರೆ.",
        "ಹಲವು ಭಾಷೆಗಳನ್ನು ಕಲಿಯುವುದು ಅರಿವಿನ ಸಾಮರ್ಥ್ಯ ಮತ್ತು ಪರಾನುಭೂತಿಯನ್ನು ಹೆಚ್ಚಿಸುತ್ತದೆ.",
        "ನಿಮ್ಮ ಸಮಯ ಮತ್ತು ಮಾರ್ಗದರ್ಶನಕ್ಕಾಗಿ ತುಂಬಾ ಧನ್ಯವಾದಗಳು."
    ]
}

directions = [
    ("en", "hi"),
    ("en", "kn"),
    ("hi", "en"),
    ("kn", "en"),
    ("hi", "kn"),
    ("kn", "hi"),
]

configs = [
    ("torch1b (beam 2)", "torch1b", "2"),
    ("torch1b (greedy)", "torch1b", "1"),
    ("onnx (int8 distilled)", "onnx", "1"),
]

benchmark_results = {}

print("==========================================================================")
print("BENCHMARK: 20 sentences per direction, 6 directions")
print("Comparing: torch1b beam 2, torch1b greedy, onnx int8 distilled")
print("==========================================================================\n")

for cfg_name, backend_mode, beam_val in configs:
    os.environ["TRANSLATOR_BACKEND"] = backend_mode
    os.environ["TR_BEAMS"] = beam_val
    tp.TRANSLATOR_BACKEND = backend_mode
    
    # Clear caches
    with tp._tr_lock:
        tp._text_cache.clear()
        tp._model_cache.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    print(f"\n>>> RUNNING CONFIG: {cfg_name} (backend={backend_mode}, beams={beam_val})")
    benchmark_results[cfg_name] = {}

    for src_iso, tgt_iso in directions:
        dir_key = f"{src_iso}->{tgt_iso}"
        sentences = test_sentences[src_iso]
        references = test_sentences[tgt_iso]
        latencies = []
        outputs = []

        # Preload model
        tp._get_model(src_iso, tgt_iso, target_backend=backend_mode)

        for text in sentences:
            t0 = time.perf_counter()
            out = tp.translate(text, src_iso, tgt_iso)
            elapsed_ms = (time.perf_counter() - t0) * 1000.0
            latencies.append(elapsed_ms)
            outputs.append(out)

        p50 = float(np.percentile(latencies, 50))
        p95 = float(np.percentile(latencies, 95))
        peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024.0 * 1024.0)) if torch.cuda.is_available() else 0.0
        
        # Compute chrF++ score
        chrf = sacrebleu.corpus_chrf(outputs, [references], word_order=2)
        score_chrf = round(chrf.score, 2)

        benchmark_results[cfg_name][dir_key] = {
            "p50_ms": round(p50, 1),
            "p95_ms": round(p95, 1),
            "peak_vram_mb": round(peak_vram_mb, 1),
            "chrf2": score_chrf,
            "sample_src": sentences[0],
            "sample_out": outputs[0],
            "sample_ref": references[0],
        }
        print(f"  [{dir_key}] p50: {p50:.1f}ms | p95: {p95:.1f}ms | Peak VRAM: {peak_vram_mb:.1f}MB | chrF++: {score_chrf}")

# Print Summary Table
print("\n" + "=" * 95)
print(f"{'Direction':<10} | {'Config':<22} | {'p50 (ms)':<10} | {'p95 (ms)':<10} | {'VRAM (MB)':<10} | {'chrF++':<8}")
print("-" * 95)
for src_iso, tgt_iso in directions:
    dir_key = f"{src_iso}->{tgt_iso}"
    for cfg_name, _, _ in configs:
        res = benchmark_results[cfg_name][dir_key]
        print(f"{dir_key:<10} | {cfg_name:<22} | {res['p50_ms']:<10} | {res['p95_ms']:<10} | {res['peak_vram_mb']:<10} | {res['chrf2']:<8}")
    print("-" * 95)

# Eyeball Sample Outputs Side-by-Side
print("\n" + "=" * 95)
print("EYEBALL SAMPLE OUTPUTS SIDE-BY-SIDE")
print("=" * 95)
for src_iso, tgt_iso in directions:
    dir_key = f"{src_iso}->{tgt_iso}"
    sample_src = test_sentences[src_iso][0]
    sample_ref = test_sentences[tgt_iso][0]
    print(f"\nDirection: {dir_key}")
    print(f"  Source:    {sample_src}")
    print(f"  Reference: {sample_ref}")
    for cfg_name, _, _ in configs:
        out = benchmark_results[cfg_name][dir_key]["sample_out"]
        print(f"  {cfg_name:<22}: {out}")

# Recommendation Analysis
print("\n" + "=" * 95)
print("RECOMMENDATION DECISION")
print("=" * 95)
all_p95_torch1b_b2 = [benchmark_results["torch1b (beam 2)"][f"{s}->{t}"]["p95_ms"] for s, t in directions]
all_p95_torch1b_greedy = [benchmark_results["torch1b (greedy)"][f"{s}->{t}"]["p95_ms"] for s, t in directions]
max_p95_b2 = max(all_p95_torch1b_b2)
max_p95_greedy = max(all_p95_torch1b_greedy)

if max_p95_b2 <= 500.0:
    print(f"✅ RECOMMENDATION: Keep IndicTrans2 1B (beam 2).")
    print(f"   Reason: Worst-case p95 latency is {max_p95_b2:.1f}ms (<= 500ms target) with highest translation fidelity.")
elif max_p95_greedy <= 500.0:
    print(f"✅ RECOMMENDATION: Keep IndicTrans2 1B with greedy decoding (TR_BEAMS=1).")
    print(f"   Reason: Beam 2 p95 exceeded 500ms ({max_p95_b2:.1f}ms), but greedy p95 is {max_p95_greedy:.1f}ms (<= 500ms target)")
    print(f"   while retaining 1B parameter contextual translation superior to distilled ONNX models.")
else:
    print(f"⚠️ RECOMMENDATION: Automatic fallback to ONNX int8 distilled recommended for latency-critical paths.")
    print(f"   Reason: Both 1B configurations exceeded 500ms p95 on T4 GPU.")


In [ ]:
# 8. Start pyngrok tunnel to proxy port 8765
import os
import subprocess
import time
import json
import urllib.request
from google.colab import userdata
from pyngrok import conf, ngrok

token = userdata.get('NGROK_AUTHTOKEN')
if token:
    os.environ["NGROK_AUTHTOKEN"] = token
    conf.get_default().auth_token = token
    print("✅ Loaded NGROK_AUTHTOKEN from Colab Secrets.")

# Ensure ngrok binary is installed locally
try:
    ngrok_bin = conf.get_default().ngrok_path
    if not os.path.exists(ngrok_bin):
        ngrok.install_ngrok()
except Exception:
    ngrok.install_ngrok()

!pkill -9 -f ngrok || true
ngrok.kill()
time.sleep(2)

ngrok_bin = conf.get_default().ngrok_path
subprocess.Popen([
    ngrok_bin, "http", "8765", "--domain=aspire-reluctant-obsession.ngrok-free.dev"
])
time.sleep(5)

try:
    req = urllib.request.urlopen("http://127.0.0.1:4040/api/tunnels")
    data = json.loads(req.read().decode())
    url = data['tunnels'][0]['public_url']
    print("✅ Ngrok is officially ONLINE!")
    print("WebSocket URL for Client:", url.replace("https://", "wss://"))
except Exception as e:
    print("Could not retrieve active tunnel:", e)


In [ ]:
# 9. Start bridge (8766), wait, start proxy (8765), and print nvidia-smi memory
!fuser -k 8765/tcp 8766/tcp || true
!pkill -f "[a]pp_bridge.py" || true
!pkill -f "[t]ranslation_proxy.py" || true
!sleep 2

# Bridge on 8766 (ASR only)
!cd /content/subtitlesssss && QT_QPA_PLATFORM=offscreen setsid nohup python backend/app_bridge.py --host 127.0.0.1 --model distil-small.en --port 8766 > server.log 2>&1 < /dev/null &
print("Starting app_bridge.py on 8766, waiting 20s...")
!sleep 20

# Proxy on 8765 (Translation layer)
!cd /content/subtitlesssss && setsid nohup python translation_proxy.py > proxy.log 2>&1 < /dev/null &
print("Starting translation_proxy.py on 8765, waiting 5s...")
!sleep 5

!grep -v "Deprecat\|HTTP Request" /content/subtitlesssss/server.log | tail -n 8
!tail -n 8 /content/subtitlesssss/proxy.log
!(ss -ltnp | grep -E "8765|8766") || echo "ports not listening"

print("\n--- GPU Memory Usage (nvidia-smi) ---")
!nvidia-smi --query-gpu=memory.used,memory.total,memory.free --format=csv


In [ ]:
# 10. STEP 5: End-to-End WebSocket Contract Verification (ws://127.0.0.1:8765)
import asyncio
import json
import websockets

PROXY_WS = "ws://127.0.0.1:8765"

async def test_handshake_and_subtitles(asr_lang, tgt_lang, test_speech_text):
    print(f"\n--- Testing WebSocket flow: asr_language={asr_lang} -> target_lang={tgt_lang} ---")
    async with websockets.connect(PROXY_WS, ping_timeout=10) as ws:
        # 1. Send Handshake
        handshake = {
            "type": "HANDSHAKE",
            "sampleRate": 16000,
            "numChannels": 1,
            "asrConfig": {
                "asr_language": asr_lang,
                "tgt_lang": tgt_lang,
            }
        }
        await ws.send(json.dumps(handshake))
        print("Sent HANDSHAKE:", handshake)

        # 2. Simulate upstream final subtitle frame
        simulated_sub = {
            "type": "final",
            "text": test_speech_text,
            "language": asr_lang,
            "is_final": True,
            "start": 0.0,
            "end": 2.5,
            "processing_time_s": 0.35,
        }
        await ws.send(json.dumps(simulated_sub))
        
        # 3. Receive and validate proxy response
        msg = await asyncio.wait_for(ws.recv(), timeout=15.0)
        reply = json.loads(msg)
        print("Received Reply from Proxy:", reply)

        # Validate contract shape
        assert "text" in reply, "Missing 'text' key in response"
        assert "source_text" in reply, "Missing 'source_text' key in response"
        assert "language" in reply, "Missing 'language' key in response"
        assert reply["language"] == tgt_lang, f"Expected language={tgt_lang}, got {reply['language']}"
        assert reply["source_text"] == test_speech_text, f"Expected source_text='{test_speech_text}'"
        print(f"✅ Verified contract: text='{reply['text']}', source_text='{reply['source_text']}', language='{reply['language']}'")

async def run_e2e_tests():
    # Test 1: kn -> en
    await test_handshake_and_subtitles("kn", "en", "ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?")
    
    # Test 2: en -> kn
    await test_handshake_and_subtitles("en", "kn", "Hello, how are you today?")
    
    print("\n========================================================")
    print("✅ ALL END-TO-END WEBSOCKET CONTRACT CHECKS PASSED!")
    print("========================================================")

asyncio.run(run_e2e_tests())


In [ ]:
!grep "SUB \|translation layer\|new upstream\|error" /content/subtitlesssss/proxy.log | tail -n 25
